In [1]:
print("allok")

allok


In [1]:
import os
import getpass
import pandas as pd

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore

from deepeval import evaluate
from deepeval.dataset import EvaluationDataset, Golden
from deepeval.test_case import LLMTestCase

In [3]:
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")


In [4]:
RAG_MODEL = "gpt-4.1-mini"
EMBEDDING_MODEL ="text-embedding-3-small"
DEEPEVAL_JUDGE_MODEL = "gpt-4.1-mini"

In [5]:
os.environ[
    "DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE"
] = "300"

os.environ[
    "DEEPEVAL_PER_TASK_TIMEOUT_SECONDS_OVERRIDE"
] = "600"

os.environ[
    "DEEPEVAL_RETRY_MAX_ATTEMPTS"
] = "1"

In [6]:
documents = [
    Document(
        page_content="Full-time employees receive 24 paid leaves per calendar year.",
        metadata={"doc_id": "leave_policy"},
    ),
    Document(
        page_content="Employees are allowed to work from home for a maximum of 2 days per week.",
        metadata={"doc_id": "remote_policy"},
    ),
    Document(
        page_content="Employees can claim up to ₹3000 per month for internet reimbursement.",
        metadata={"doc_id": "internet_policy"},
    ),
    Document(
        page_content="The standard probation period for new employees is 6 months.",
        metadata={"doc_id": "probation_policy"},
    ),
    Document(
        page_content="Employees receive ₹1000 per month as mobile reimbursement.",
        metadata={"doc_id": "mobile_policy"},
    ),
    Document(
        page_content="Medical insurance coverage begins from the employee's date of joining.",
        metadata={"doc_id": "insurance_policy"},
    ),
]


In [7]:
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

In [8]:
vector_store = InMemoryVectorStore(embedding=embeddings)

In [9]:
vector_store.add_documents(documents)

['1a963067-e3dc-489f-9034-82db2c0f79d2',
 'c18e93b3-d86e-4ddc-b274-c294566a711d',
 'fffec764-0ff8-4c62-ba42-28c347a245d4',
 '92abd987-396b-4a25-8166-62faa512cdbe',
 '47968daf-893f-4bfe-a914-efab501c77fa',
 '8c3a0a5e-d6ac-4250-a276-25e6c06d50ed']

In [10]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

In [11]:
llm = ChatOpenAI(
    model=RAG_MODEL,
    temperature=0,
)

In [12]:
def rag_pipeline(query: str) -> dict:
    retrieved_docs = retriever.invoke(query)

    retrieval_context = [
        doc.page_content
        for doc in retrieved_docs
    ]

    retrieved_doc_ids = [
        doc.metadata.get("doc_id")
        for doc in retrieved_docs
    ]

    context = "\n\n".join(retrieval_context)

    prompt = f"""
You are an HR policy assistant.

Answer the user's question ONLY from the supplied context.

Rules:
1. Do not use outside knowledge.
2. Do not invent policy details.
3. If the answer is not present in the context, say:
   "I don't know based on the provided context."
4. Keep the answer concise.

CONTEXT:
{context}

QUESTION:
{query}
"""

    response = llm.invoke(prompt)

    return {
        "answer": response.content,
        "retrieval_context": retrieval_context,
        "retrieved_doc_ids": retrieved_doc_ids,
    }


In [13]:
sample = rag_pipeline("What is the monthly internet reimbursement limit?")

print("ANSWER:")
print(sample["answer"])

print("\nRETRIEVED DOCS:")
print(sample["retrieved_doc_ids"])

print("\nCONTEXT:")
for chunk in sample["retrieval_context"]:
    print("-", chunk)

ANSWER:
The monthly internet reimbursement limit is ₹3000.

RETRIEVED DOCS:
['internet_policy', 'mobile_policy', 'remote_policy']

CONTEXT:
- Employees can claim up to ₹3000 per month for internet reimbursement.
- Employees receive ₹1000 per month as mobile reimbursement.
- Employees are allowed to work from home for a maximum of 2 days per week.
